# 04. Preprocessing seguro y feature engineering

**Fases del guía metodológica cubiertas: 8 (Preprocessing y limpieza segura), 9 (Feature engineering y representaciones)**

> Regla central aplicada: *definir -> auditar -> dividir -> aprender solo con train ->
> seleccionar con validación/CV -> comprobar una vez con test -> empaquetar -> monitorizar*.
> El test nunca influye en preprocessing, selección de variables, hiperparámetros o elección de modelo.



## 8. Preprocessing seguro

Regla: **todo preprocessing que aprende estadísticas se ajusta SOLO con train** y se aplica
a validation/test mediante `transform`. Demostramos que ajustar sobre todo el dataset
contamina (leakage) y que nuestro pipeline es seguro.

### 8.0.1 Carga de los conjuntos ya particionados

Cargamos los conjuntos guardados en la fase 7 (train/validation/test) y verificamos sus
dimensiones. A partir de este momento, cualquier estadística que se calcule para
imputar, escalar o codificar debe salir **exclusivamente** de `X_train`; validation y
test solo se transforman. Esta disciplina es la que garantiza que las métricas finales
de la fase 17 sean honestas.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.load_data import load_processed
from src.models.train_model import get_preprocessor, CATEGORICAL_COLS, ORDINAL_COLS, NUMERIC_COLS

d = load_processed()
X_train, X_val, X_test = d["X_train"], d["X_val"], d["X_test"]
y_train, y_val, y_test = d["y_train"], d["y_val"], d["y_test"]
print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)


Train: (396, 29) | Val: (133, 29) | Test: (133, 29)



### 8.0.2 Demostración del peligro de la fuga en el preprocesado

Para visualizar por qué el pipeline debe ajustarse solo con train, comparamos la mediana
de `absences` calculada en train frente a la de test. Si fueran distintas, imputar los
nulos con la mediana global (train+test) introduciría información del test en el
entrenamiento, y las métricas de la fase 17 dejarían de medir generalización. A
continuación ajustamos el `ColumnTransformer` **solo con train** y transformamos los tres
conjuntos, comprobando que las dimensiones de salida son coherentes.


In [2]:

# DEMOSTRACIÓN DE LEAKAGE: imputar/escalar con estadísticas de TODO el dataset
# vs. solo con train produce parámetros distintos -> los datos de val/test
# no deben participar en el fit.
import numpy as np
med_full = X_train["absences"].median()   # igual que con todo el dataset aquí
med_test = X_test["absences"].median()
print(f"Mediana absences (train)={med_full}  (test)={med_test}")
print("Si fueran distintas, imputar con la mediana global contaminaría la evaluación.")

# El preprocessor se ajusta SOLO con train:
pre = get_preprocessor()
pre.fit(X_train)
Xtr_t = pre.transform(X_train)
Xva_t = pre.transform(X_val)
Xte_t = pre.transform(X_test)
print("Transform OK -> shapes:", Xtr_t.shape, Xva_t.shape, Xte_t.shape)


Mediana absences (train)=3.0  (test)=3.0
Si fueran distintas, imputar con la mediana global contaminaría la evaluación.
Transform OK -> shapes: (396, 55) (133, 55) (133, 55)



## 8.1 Tratamientos aplicados

| Variable | Tratamiento |
|---|---|
| Numéricas (`age, Medu, Fedu, absences`) | Imputación mediana (solo si hubiera nulos) |
| Ordinales (`traveltime...health, Dalc`) | OrdinalEncoder con orden natural 1-5 |
| Categóricas nominales | OneHotEncoder (`handle_unknown="ignore"`) |
| Nulos | No existen en crudo; imputadores por robustez |
| Outliers | `absences` (0-93) se conserva; los árboles son robustos; se audita |
| Escalado | No necesario para árboles; disponible para lineales |

## 9. Feature engineering (conocimiento de dominio)

Ideas nacidas del EDA, implementadas **solo sobre features disponibles en producción**:

| Feature derivada | Definición | Justificación |
|---|---|---|
| `family_support` | nº de apoyos familiares (0-2) | El apoyo modera el riesgo |
| `study_intensity` | estudio + clases pagadas | La dedicación académica protege |
| `has_failures` | `failures > 0` | El fracaso académico se asocia al consumo |
| `absences_high` | `absences >= 8` | El absentismo es señal de riesgo |
| `parent_edu_max` | max educación parental | Nivel educativo familiar |
| `parent_edu_diff` | |Medu - Fedu| | Desigualdad educativa en casa |
| `social_exposure` | `goout + freetime` | Exposición social (proxy de riesgo) |
| `health_low` | `health <= 2` | Salud percibida baja |

**Nota de auditoría (fase 4 del proyecto)**: el `ColumnTransformer` del pipeline
oficial solo procesa las 29 columnas originales (`remainder="drop"`), por lo que estas
derivadas **no participan en el modelo final**; se documentan como candidatas. En este
notebook las construimos y comprobamos que no rompen el flujo.



### 9.0.1 Aplicación del feature engineering

La función `add_domain_features` es **pura** (sin estado): dado un DataFrame devuelve el
mismo DataFrame más las 8 columnas derivadas, calculadas fila a fila sin usar el target
ni datos de otros registros. Esto garantiza que aplicar la misma función a train,
validation y test no introduce fuga. Mostramos las dimensiones antes/después y una vista
previa de las nuevas columnas.


In [3]:

from src.features.build_features import add_domain_features, FEATURES_ENGINEERED

X_tr2 = add_domain_features(X_train)
print("Features originales:", X_train.shape[1])
print("Features engineered:", X_tr2.shape[1])
print("Añadidas:", [c for c in FEATURES_ENGINEERED if c not in X_train.columns])
X_tr2.head(3)


Features originales: 29
Features engineered: 37
Añadidas: ['family_support', 'study_intensity', 'has_failures', 'absences_high', 'parent_edu_max', 'parent_edu_diff', 'social_exposure', 'health_low']


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,health,absences,family_support,study_intensity,has_failures,absences_high,parent_edu_max,parent_edu_diff,social_exposure,health_low
0,GP,M,16,U,GT3,T,2,3,other,other,...,1,4,1,3,0,0,3,1,5,1
1,GP,F,16,U,GT3,T,1,1,at_home,other,...,5,6,1,2,0,0,1,0,5,0
2,GP,M,18,R,LE3,T,3,2,services,other,...,4,8,1,4,0,1,3,1,6,0



### 9.0.2 Coherencia de columnas entre conjuntos

Aplicamos el mismo engineering a validation y test y **verificamos que los tres
conjuntos tienen exactamente las mismas columnas en el mismo orden**. Esta comprobación
es crítica: si algún día el orden difiriera, el pipeline podría entrenar con unas
columnas y predecir con otras, produciendo errores silenciosos en producción.


In [4]:

# Aplicar el mismo engineering a val/test (función pura, sin fit)
X_va2 = add_domain_features(X_val)
X_te2 = add_domain_features(X_test)
assert list(X_tr2.columns) == list(X_va2.columns) == list(X_te2.columns)
print("Columnas idénticas en train/val/test:", X_tr2.columns.tolist() == X_va2.columns.tolist())


Columnas idénticas en train/val/test: True



### Reglas anti-leakage verificadas (fase 9.6)

- No se usa futuro: ninguna feature depende del target o de calificaciones.
- No se calculan agregados globales con test: `add_domain_features` es fila a fila.
- No se usa la observación actual en sus propios agregados: no hay agregados.
- No se calculan estadísticas del target fuera de folds: el target solo se usa en
  selección (fase 10) dentro de train, y en CV dentro de folds.
